In [ ]:
from google.colab import files

uploaded = files.upload()

Saving preprocessing_config.json to preprocessing_config (1).json
Saving road_dataset_splits.csv to road_dataset_splits (1).csv
Saving road_dataset_224.zip to road_dataset_224 (1).zip


In [ ]:
import json
import pandas as pd

# Read preprocessing configuration
with open('/content/preprocessing_config.json', 'r') as f:
    config = json.load(f)

# Read dataset information
df = pd.read_csv(
  '/content/road_dataset_splits.csv')

# Display configuration
print("Preprocessing Configuration:")
print(config)

# Display first 5 rows
print("\nFirst 5 rows:")
display(df.head())

# Check number of images in each split
print("\nSplit Distribution:")
print(df['split'].value_counts())

# Check class distribution
print("\nClass Distribution:")
display(pd.crosstab(df['split'], df['label']))

Preprocessing Configuration:
{'img_size': 224, 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'seed': 42, 'labels': {'Normal': 0, 'Damaged': 1}}

First 5 rows:


,split,image_path,original_class,label,label_id
0,train,train/Damaged/train_minor_damage_0690.jpg,Minor Damage,Damaged,1
1,train,train/Damaged/train_major_damage_1041.jpg,Major Damage,Damaged,1
2,test,test/Normal/test_normal_road_0192.jpg,Normal Road,Normal,0
3,val,val/Normal/val_normal_road_0187.jpg,Normal Road,Normal,0
4,train,train/Normal/train_normal_road_0279.jpg,Normal Road,Normal,0



Split Distribution:
split
train    3398
test      731
val       727
Name: count, dtype: int64

Class Distribution:


label,Damaged,Normal
split,,
test,531,200
train,2470,928
val,529,198


In [ ]:
import zipfile
import os

zip_path = '/content/road_dataset_224.zip'
extract_dir = '/content/road_dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extraction completed!")

print(os.listdir(extract_dir)[:10])

Extraction completed!
['train', 'test', 'val']


In [ ]:
import os

for split in ['train', 'val', 'test']:
    split_path = f'/content/road_dataset/{split}'

    print(f"\n{split.upper()}:")

    for class_name in ['Normal', 'Damaged']:
        class_path = os.path.join(split_path, class_name)
        images = os.listdir(class_path)

        print(f"{class_name}: {len(images)} images")


TRAIN:
Normal: 928 images
Damaged: 2470 images

VAL:
Normal: 198 images
Damaged: 529 images

TEST:
Normal: 200 images
Damaged: 531 images


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets

# Image normalization
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Load datasets
train_dataset = datasets.ImageFolder(
    '/content/road_dataset/train',
    transform=transform
)

val_dataset = datasets.ImageFolder(
    '/content/road_dataset/val',
    transform=transform
)

test_dataset = datasets.ImageFolder(
    '/content/road_dataset/test',
    transform=transform
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Class mapping:", train_dataset.class_to_idx)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Class mapping: {'Damaged': 0, 'Normal': 1}
Train batches: 107
Validation batches: 23
Test batches: 23


In [ ]:
import os

# Our agreed label mapping
class_mapping = {
    'Normal': 0,
    'Damaged': 1
}

# Update all datasets
for dataset in [train_dataset, val_dataset, test_dataset]:

    dataset.class_to_idx = class_mapping

    dataset.samples = [
        (path, class_mapping[os.path.basename(os.path.dirname(path))])
        for path, old_label in dataset.samples
    ]

    dataset.imgs = dataset.samples
    dataset.targets = [label for _, label in dataset.samples]

print("Updated mapping:", train_dataset.class_to_idx)
print("Sample labels:", train_dataset.targets[:10])

Updated mapping: {'Normal': 0, 'Damaged': 1}
Sample labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels:", labels[:10])
print("Unique labels:", labels.unique())

Images shape: torch.Size([32, 3, 224, 224])
Labels: tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1])
Unique labels: tensor([0, 1])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [ ]:
class CustomCNN(nn.Module):

    def __init__(self):
        super(CustomCNN, self).__init__()

        # First convolution block
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)

        # Second convolution block
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Third convolution block
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # Fourth convolution block
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(256 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.fc2(x)

        return x

In [ ]:
model = CustomCNN().to(device)

print(model)

CustomCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=50176, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
)


In [ ]:
images, labels = next(iter(train_loader))

images = images.to(device)

output = model(images)

print("Output shape:", output.shape)

Output shape: torch.Size([32, 2])


In [ ]:
#Calculate class weights from training data only
normal_count = train_dataset.targets.count(0)
damaged_count = train_dataset.targets.count(1)
total = normal_count + damaged_count
normal_weight = total / (2 * normal_count)
damaged_weight = total / (2 * damaged_count)
class_weights = torch.tensor( [normal_weight, damaged_weight], dtype=torch.float32 ).to(device)
criterion = nn.CrossEntropyLoss( weight=class_weights )
print("Class Weights:")
print(f"Normal  : {normal_weight:.4f}")
print(f"Damaged : {damaged_weight:.4f}")

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Training settings
num_epochs = 15

print("Loss Function:", criterion)
print("Optimizer:", optimizer)
print("Epochs:", num_epochs)

Class Weights:
Normal  : 1.8308
Damaged : 0.6879
Loss Function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Epochs: 15


In [ ]:
import time

train_losses = []
val_losses = []

for epoch in range(num_epochs):

    # Training phase
    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # Validation phase
    model.eval()

    val_running_loss = 0.0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

    epoch_val_loss = val_running_loss / len(val_loader)
    val_losses.append(epoch_val_loss)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Val Loss: {epoch_val_loss:.4f}"
    )

Epoch [1/15] Train Loss: 0.5285 Val Loss: 0.4120
Epoch [2/15] Train Loss: 0.3930 Val Loss: 0.3402
Epoch [3/15] Train Loss: 0.3494 Val Loss: 0.3287
Epoch [4/15] Train Loss: 0.3333 Val Loss: 0.3126
Epoch [5/15] Train Loss: 0.2967 Val Loss: 0.3360
Epoch [6/15] Train Loss: 0.2631 Val Loss: 0.2423
Epoch [7/15] Train Loss: 0.2297 Val Loss: 0.2954
Epoch [8/15] Train Loss: 0.2055 Val Loss: 0.2952
Epoch [9/15] Train Loss: 0.2068 Val Loss: 0.3519
Epoch [10/15] Train Loss: 0.1555 Val Loss: 0.2627
Epoch [11/15] Train Loss: 0.1331 Val Loss: 0.2472


In [ ]:
import copy

# Reset model and optimizer
model = CustomCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

best_val_loss = float('inf')
best_epoch = 0

train_losses = []
val_losses = []

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

    epoch_val_loss = val_running_loss / len(val_loader)
    val_losses.append(epoch_val_loss)

    # Save best model
    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss
        best_epoch = epoch + 1

        torch.save(model.state_dict(), '/content/best_custom_cnn.pth')

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Val Loss: {epoch_val_loss:.4f}"
    )

print(f"\nBest Epoch: {best_epoch}")
print(f"Best Validation Loss: {best_val_loss:.4f}")

NameError: name 'CustomCNN' is not defined

In [ ]:
model.load_state_dict(
    torch.load('/content/best_custom_cnn.pth', map_location=device)
)

model.eval()

print("Best model loaded successfully!")

Best model loaded successfully!


In [ ]:
from google.colab import files

files.download('/content/best_custom_cnn.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())


# Calculate metrics
accuracy = accuracy_score(all_labels, all_predictions)

precision = precision_score(
    all_labels, all_predictions,
    pos_label=1
)

recall = recall_score(
    all_labels, all_predictions,
    pos_label=1
)

f1 = f1_score(
    all_labels, all_predictions,
    pos_label=1
)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=['Normal', 'Damaged']
))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

Accuracy: 0.9001
Precision: 0.9194
Recall: 0.9454
F1-score: 0.9322

Classification Report:
              precision    recall  f1-score   support

      Normal       0.84      0.78      0.81       200
     Damaged       0.92      0.95      0.93       531

    accuracy                           0.90       731
   macro avg       0.88      0.86      0.87       731
weighted avg       0.90      0.90      0.90       731

Confusion Matrix:
[[156  44]
 [ 29 502]]


In [ ]:
import os
import torch
import numpy as np
from PIL import ImageEnhance
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

# Change brightness before converting the image to a tensor
class AdjustBrightness:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, image):
        return ImageEnhance.Brightness(image).enhance(self.factor)


# Add Gaussian noise before normalization
class AddGaussianNoise:
    def __init__(self, std):
        self.std = std

    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * self.std
        return torch.clamp(tensor + noise, 0, 1)


mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

class_mapping = {'Normal': 0, 'Damaged': 1}


def create_test_loader(brightness=1.0, noise_std=0.0):
    transform_list = []

    if brightness != 1.0:
        transform_list.append(AdjustBrightness(brightness))

    transform_list.append(transforms.ToTensor())

    if noise_std > 0:
        transform_list.append(AddGaussianNoise(noise_std))

    transform_list.append(transforms.Normalize(mean, std))

    transform = transforms.Compose(transform_list)

    dataset = datasets.ImageFolder(
        '/content/road_dataset/test',
        transform=transform
    )

    dataset.class_to_idx = class_mapping
    dataset.samples = [
        (path, class_mapping[os.path.basename(os.path.dirname(path))])
        for path, _ in dataset.samples
    ]
    dataset.imgs = dataset.samples
    dataset.targets = [label for _, label in dataset.samples]

    return DataLoader(dataset, batch_size=32, shuffle=False)


print("Robustness testing setup is ready!")

Robustness testing setup is ready!


In [ ]:
model.load_state_dict(
    torch.load('/content/best_custom_cnn.pth', map_location=device)
)
model.eval()

print("Best Custom CNN checkpoint loaded!")

Best Custom CNN checkpoint loaded!


In [ ]:
def evaluate_model(loader):
    all_predictions = []
    all_labels = []

    model.eval()

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions, pos_label=1)

    return accuracy, f1


brightness_levels = [0.7, 1.0, 1.3]

for level in brightness_levels:
    loader = create_test_loader(brightness=level)
    accuracy, f1 = evaluate_model(loader)

    print(f"Brightness: {level}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("-" * 30)

Brightness: 0.7
Accuracy: 0.8317
F1-score: 0.8897
------------------------------
Brightness: 1.0
Accuracy: 0.9001
F1-score: 0.9322
------------------------------
Brightness: 1.3
Accuracy: 0.8974
F1-score: 0.9307
------------------------------


In [ ]:
noise_levels = [0.0, 0.05, 0.10, 0.20]

noise_results = []

for level in noise_levels:
    loader = create_test_loader(noise_std=level)
    accuracy, f1 = evaluate_model(loader)

    noise_results.append({
        "Noise Level": level,
        "Accuracy": accuracy,
        "F1-score": f1
    })

    print(f"Noise: {level}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("-" * 30)

Noise: 0.0
Accuracy: 0.9001
F1-score: 0.9322
------------------------------
Noise: 0.05
Accuracy: 0.8249
F1-score: 0.8891
------------------------------
Noise: 0.1
Accuracy: 0.7278
F1-score: 0.8422
------------------------------
Noise: 0.2
Accuracy: 0.7264
F1-score: 0.8415
------------------------------
